
# ModernBERT Classifier Demo

This notebook fine-tunes `answerdotai/ModernBERT-base` as a tiny binary classifier.

Demo task:

- `luna` = simple / bounded software task
- `astra` = complex / security-sensitive / architectural task

The goal is to show how a small encoder model can replace an LLM for a narrow classification task.

> In Colab, enable a GPU from **Runtime → Change runtime type → T4 GPU** before training.


In [ ]:

!pip -q install -U "transformers>=4.48.0" datasets accelerate scikit-learn


## 1. Create a tiny demo dataset

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split

examples = [
    # Luna: simple / bounded
    ("Fix a typo in the README.", 0),
    ("Change the primary button color from blue to green.", 0),
    ("Rename a local variable for clarity.", 0),
    ("Add a missing unit test for an existing helper.", 0),
    ("Update an API error message.", 0),
    ("Add pagination to an existing endpoint.", 0),
    ("Refactor duplicate validation logic into a shared helper.", 0),
    ("Add structured logging around failed HTTP requests.", 0),
    ("Update the OAuth login page copy without changing authentication logic.", 0),
    ("Rename the authentication middleware function without changing behavior.", 0),
    ("Add a runtime-only field to the User object that is not persisted.", 0),
    ("Add another field to an API response using data already in memory.", 0),
    ("Improve the wording of a validation error.", 0),
    ("Add a new test case for an existing endpoint.", 0),
    ("Document how to run the development server.", 0),
    ("Extract a small helper function without changing behavior.", 0),

    # Astra: complex / risky
    ("Implement Google OAuth login and persist users in PostgreSQL.", 1),
    ("Add refresh token rotation and revocation.", 1),
    ("Migrate user IDs from integers to UUIDs.", 1),
    ("Replace session authentication with JWT authentication.", 1),
    ("Add role-based access control for admins and normal users.", 1),
    ("Create an audit log table and record security-sensitive user actions.", 1),
    ("Add an encrypted API key field to the User model and persist it.", 1),
    ("Redesign the authorization layer across multiple services.", 1),
    ("Implement password reset using signed expiring tokens.", 1),
    ("Add multi-tenant permissions and migrate existing authorization data.", 1),
    ("Change token issuance rules and refresh-token storage.", 1),
    ("Move authentication state from database-backed sessions to stateless tokens.", 1),
    ("Add a new persisted field to User and write the required database migration.", 1),
    ("Introduce an event-driven workflow across the API and worker services.", 1),
    ("Split the monolith authentication module into separate services.", 1),
    ("Add end-to-end encryption for stored customer credentials.", 1),
]

df = pd.DataFrame(examples, columns=["text", "label"])

train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=df["label"],
)

print("Train:", len(train_df))
print("Test:", len(test_df))
display(train_df.head())


## 2. Load ModernBERT

In [ ]:

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

MODEL_NAME = "answerdotai/ModernBERT-base"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "luna", 1: "astra"},
    label2id={"luna": 0, "astra": 1},
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)


## 3. Fine-tune the classifier

In [ ]:

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./modernbert-demo",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=5,
    report_to="none",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


## 4. Evaluate

In [ ]:

metrics = trainer.evaluate()
metrics


## 5. Classify new tasks

In [ ]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def classify(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits[0]

    probs = torch.softmax(logits, dim=-1)

    result = {
        model.config.id2label[i]: float(probs[i])
        for i in range(len(probs))
    }

    prediction = max(result, key=result.get)

    return {
        "prediction": prediction,
        "confidence": result[prediction],
        "probabilities": result,
    }

classify(
    "Implement OAuth authentication and add a PostgreSQL migration for refresh tokens."
)


## 6. Try some differently-worded examples

In [ ]:

tests = [
    "Please clean up the docs and correct a spelling mistake.",
    "Users should be able to sign in with Google and their linked identity must be stored.",
    "Add a computed property to User that only exists while the app is running.",
    "Change how refresh tokens are generated, stored, and revoked.",
    "Add one more assertion to the existing test suite.",
]

for text in tests:
    result = classify(text)
    print("\n", text)
    print(result)



## What to do next

This dataset is intentionally tiny, so treat the result as a functional demo rather than a meaningful benchmark.

For a real experiment:

1. Collect **500–5,000+** labeled examples.
2. Keep paraphrases from the same underlying task in the same train/test split.
3. Add lots of **hard negatives**.
4. Evaluate on genuinely unseen wording.
5. Add probability calibration before using confidence thresholds in production.

You can reuse the same notebook for email intent classification by replacing the dataset and labels.
